# FR02
* **Numerical (Số)**: `budget`, `revenue`, `runtime`, `vote_average`, `vote_count`.
* **Categorical (Phân loại)**: `genres`, `production_companies`.
* **Date/Time (Ngày/Giờ)**: `release_date`.
* **Identifier (Định danh)**: `id`.
* **Free-text / Links**: `title`, `overview`, .

# FR03 - Define Three Hypotheses

## Hypothesis 1

* **Hypothesis Statement:** Mức độ tương quan thuận giữa Tổng số lượt đánh giá (`vote_count`) và Doanh thu (`revenue`) mạnh mẽ hơn đáng kể so với mức độ tương quan giữa Điểm đánh giá trung bình (`vote_average`) và Doanh thu (`revenue`).
* **Variables:** 
  * `vote_count` (Số nguyên)
  * `vote_average` (Số thực)
  * `revenue` (Số thực)
* **Population / Subset:** Các bộ phim có báo cáo doanh thu và lượt bình chọn hợp lệ (`revenue > 0` và `vote_count > 10`).
* **Metric:** Hệ số tương quan Pearson (r) hoặc Spearman (rs).
* **Planned Analysis:** Tính hai hệ số tương quan:
  * r1 = correlation(vote_count, revenue)
  * r2 = correlation(vote_average, revenue)
  Sau đó so sánh độ lớn |r1| và |r2|.
* **Planned Visualization:** Biểu đồ phân tán (**Scatter Plot**) kèm đường xu hướng (**Trendline**) cho từng cặp biến, hoặc **Correlation Heatmap**.
* **Decision Rule:**
  * **Accepted:** Nếu r1 > r2 và mức chênh lệch có ý nghĩa thống kê (p < 0.05).
  * **Rejected:** Nếu r1 <= r2.
  * **Inconclusive:** Nếu dữ liệu bị lệch quá nặng (right-skewed) hoặc giá trị ngoại lệ (outliers) làm méo lệch hệ số tương quan ngay cả khi đã biến đổi log.

## Giả thuyết 3

* **Phát biểu giả thuyết:** Trong các phim đã phát hành, có ngày phát hành hợp lệ và có doanh thu được báo cáo (`revenue > 0`), doanh thu trung bình của nhóm phát hành vào mùa cao điểm (tháng 5, 6, 7, 11 và 12) cao hơn ít nhất 40% so với nhóm phát hành vào các tháng còn lại.
* **Các biến sử dụng:**
  * `release_date`, `release_year`, `release_month`: xác định thời điểm phát hành.
  * `revenue`: doanh thu của phim, đơn vị USD.
  * `status`: xác định phim đã phát hành.
  * `id`: nhận diện và đếm số phim duy nhất.
* **Dữ liệu đầu vào từ FR04:** ID đã duy nhất, ngày phát hành đã hợp lệ, đồng thời `release_year` và `release_month` đã được tạo. H3 không làm sạch lại các nội dung này.
* **Quần thể phân tích:** Phim có `status == "Released"`, ngày phát hành không vượt quá ngày chốt 23/09/2026 và `revenue > 0`.
* **Chỉ số chính:** Phần trăm chênh lệch doanh thu trung bình của nhóm Cao điểm so với nhóm Thấp điểm. Trung vị và trung bình cắt ngọn 1% được dùng để kiểm tra độ nhạy với giá trị ngoại lệ.
* **Kế hoạch phân tích:** Chia phim thành hai nhóm mùa, sau đó so sánh số phim, trung bình, trung vị và trung bình cắt ngọn. Kiểm tra bổ sung theo tháng và theo thập niên.
* **Biểu đồ dự kiến:** Biểu đồ cột so sánh doanh thu trung bình của hai nhóm và biểu đồ cột theo 12 tháng.
* **Quy tắc ra quyết định:**
  * **Chấp nhận:** Mỗi nhóm có ít nhất 30 phim, mức tăng doanh thu trung bình đạt ít nhất 40% và mức tăng trung bình cắt ngọn không đảo chiều sang âm.
  * **Bác bỏ:** Hai nhóm đủ cỡ mẫu nhưng mức tăng doanh thu trung bình thấp hơn 40%.
  * **Chưa đủ bằng chứng:** Không đủ cỡ mẫu hoặc trung bình đạt ngưỡng nhưng trung bình cắt ngọn đảo chiều.

Phân tích chỉ kiểm tra mối liên hệ, không khẳng định mùa phát hành trực tiếp gây ra doanh thu cao hơn.
<!-- H3_RELEASE_SEASON_NOTEBOOK_SECTION -->

# FR04 - Prepare and Clean Data

## 1. Data Quality Assessment & General Cleaning Rules
To build a high-quality analytical base dataset, raw dataset `movies.csv` (769,631 records) underwent systematic data quality profiling and cleaning:

1. **Deduplication (`id`)**: 107,548 duplicate entity records were dropped.
2. **Missing Essential Metadata (`title`)**: 6 records lacking titles were eliminated.
3. **Datetime Validation (`release_date`)**: 59,455 records with missing/invalid release dates were removed, standardizing valid dates to `datetime64`.
4. **Impossible Financial Values Filter (`revenue`, `budget`)**: 1 record with impossible negative financial values (`revenue < 0` or `budget < 0`) was dropped.
5. **Categorical Imputation**: Missing values in `genres`, `production_companies`, `spoken_languages`, and `production_countries` were filled with `'Unknown'`.
6. **Data Availability Indicator Flags**: Added `has_revenue_data` (`revenue > 0`) and `has_budget_data` (`budget > 0`) to distinguish reported financial data from missing/unreported values (`0`).
7. **Time Feature Extraction**: Extracted `release_year` and `release_month` for downstream temporal exploratory analyses.

## 2. Data Quality Traceability Matrix
The step-by-step cleaning operations and record flow are exported to `outputs/tables/data_quality_trace.csv`.

In [ ]:
import sys
import os
import pandas as pd
import numpy as np

# Ensure src directory is in sys path
sys.path.append(os.path.abspath('../src'))
from data_cleaner import clean_movie_data

# Load raw dataset
raw_path = '../data/raw/movies.csv'
df_raw = pd.read_csv(raw_path)

# Execute general data cleaning pipeline
df_clean, df_trace = clean_movie_data(df_raw)

# Save general cleaned dataset & traceability table
os.makedirs('../data/processed', exist_ok=True)
os.makedirs('../outputs/tables', exist_ok=True)

df_clean.to_csv('../data/processed/dataset_clean.csv', index=False)
df_trace.to_csv('../outputs/tables/data_quality_trace.csv', index=False)

print('=== DATA QUALITY TRACE SUMMARY ===')
print(df_trace.to_string(index=False))

print('\n=== GENERAL CLEANED DATASET SUMMARY ===')
print(f'Original Raw Shape : {df_raw.shape}')
print(f'General Cleaned Shape: {df_clean.shape}')


# FR05 - Analyze Each Hypothesis

## Hypothesis 1

## 1. Analytical Method & Data Scope
Pursuant to the defined **Hypothesis 1 (GH1)** in FR03, we evaluate whether the linear association between total rating volume (`vote_count`) and financial revenue (`revenue`) is significantly stronger than that between average rating score (`vote_average`) and revenue.

- **Target Population Subset**: Movies with valid reported revenue (`revenue > 0`) and sufficient vote evaluation volume (`vote_count >= 10`), yielding $N = 11,622$ records.
- **Log Transformation**: Applied $\log_{10}$ transformation to `revenue` and `vote_count` to mitigate right-skewness and extreme box-office outliers.
- **Analytical Operations**: Computed Pearson correlation coefficient ($r$) across raw and log-transformed scales.

## 2. Key Findings & Sensitivity Analysis
- **Correlation Comparison**: The raw Pearson correlation for `vote_count` vs `revenue` ($r = 0.7662$) is substantially stronger than `vote_average` vs `revenue` ($r = 0.1781$).
- **Log Scale Robustness**: On log-log transformed scale, $\text{corr}(\log_{10}(\text{vote\_count}), \log_{10}(\text{revenue})) = 0.6115$, confirming that popularity volume strongly aligns with commercial revenue.
- **Potential Biases**: Heavy right-skewness and survivor bias (only movies with reported non-zero revenues are included).

In [ ]:
import sys
import os
import pandas as pd

# Ensure src directory is in sys path
sys.path.append(os.path.abspath('../src'))
from analyzer import analyze_hypothesis_1

# Load cleaned dataset from FR04
df_clean = pd.read_csv('../data/processed/dataset_clean.csv')

# Perform FR05 Analysis for Hypothesis 1
df_gh1, df_summary_gh1 = analyze_hypothesis_1(df_clean)

# Save summary results table
os.makedirs('../outputs/tables', exist_ok=True)
df_summary_gh1.to_csv('../outputs/tables/hypothesis_results.csv', index=False)

print('=== FR05 ANALYTICAL RESULTS TABLE (HYPOTHESIS 1) ===')
print(df_summary_gh1.to_string(index=False))


## Giả thuyết 3: Mùa phát hành và doanh thu

### 1. Phương pháp và phạm vi phân tích

Giả thuyết 3 nhận trực tiếp bộ dữ liệu sạch chung do FR04 tạo ra. H3 không xử lý lại ID lặp, ngày sai hoặc tạo lại `release_year`, `release_month`; thay vào đó, H3 kiểm tra các điều kiện này trước khi phân tích.

Các điều kiện lọc riêng của H3 gồm:

- Phim có `status == "Released"`.
- Ngày phát hành không vượt quá 23/09/2026.
- Doanh thu được báo cáo và lớn hơn 0.

Sau khi lọc, H3 tạo thêm `release_decade` và `season_group` vì đây là hai cột chỉ phục vụ phân tích mùa phát hành.

### 2. Kết quả phân tích đã hoàn thành

Tập phân tích có **14.839 phim duy nhất**: 8.984 phim Thấp điểm và 5.855 phim Cao điểm. Doanh thu trung bình lần lượt khoảng 32,91 triệu USD và 64,13 triệu USD, tương ứng mức tăng khoảng **94,88%**. Mức tăng trung vị khoảng 57,59% và mức tăng trung bình cắt ngọn khoảng 108,67%.

In [ ]:
import os
import sys
from pathlib import Path

import pandas as pd

# Notebook chỉ điều phối và hiển thị; logic H3 nằm trong package riêng.
sys.path.append(os.path.abspath("../src"))

from hypotheses.hypothesis_3.analyzer import (
    create_decade_check,
    create_month_summary,
    create_season_summary,
    create_trimmed_mean_by_group,
    evaluate_hypothesis_3,
)
from hypotheses.hypothesis_3.data_preparer import (
    prepare_hypothesis_3_data,
)

h3_table_dir = Path("../outputs/tables")
h3_processed_dir = Path("../data/processed")
h3_table_dir.mkdir(parents=True, exist_ok=True)
h3_processed_dir.mkdir(parents=True, exist_ok=True)

# Đọc đúng dữ liệu sạch chung do FR04 tạo ra.
df_clean_h3 = pd.read_csv(
    "../data/processed/dataset_clean.csv",
    parse_dates=["release_date"],
)

# Chỉ lọc thêm những điều kiện riêng của Giả thuyết 3.
h3_df, h3_quality_trace, h3_input_checks = (
    prepare_hypothesis_3_data(df_clean_h3)
)

# Tạo các bảng phân tích của Giả thuyết 3.
h3_season_summary = create_season_summary(h3_df)
h3_trimmed_mean_by_group = create_trimmed_mean_by_group(h3_df)
h3_month_summary = create_month_summary(h3_df)
h3_decade_check = create_decade_check(h3_df)
h3_evaluation = evaluate_hypothesis_3(
    h3_season_summary,
    h3_trimmed_mean_by_group,
)

# Xuất file riêng của H3, không ghi đè kết quả H1 hoặc H2.
h3_df.to_csv(
    h3_processed_dir / "hypothesis3_release_season_clean.csv",
    index=False,
)
h3_quality_trace.to_csv(
    h3_table_dir / "hypothesis3_quality_trace.csv",
    index=False,
)
h3_season_summary.to_csv(
    h3_table_dir / "hypothesis3_release_season_summary.csv"
)
h3_trimmed_mean_by_group.to_csv(
    h3_table_dir / "hypothesis3_trimmed_mean.csv"
)
h3_month_summary.to_csv(
    h3_table_dir / "hypothesis3_month_summary.csv",
    index=False,
)
h3_decade_check.to_csv(
    h3_table_dir / "hypothesis3_decade_check.csv"
)
h3_evaluation.to_csv(
    h3_table_dir / "hypothesis3_evaluation.csv",
    index=False,
)

# Đổi tên cột khi hiển thị để phần trình bày dùng tiếng Việt.
h3_input_checks_vi = pd.Series(
    h3_input_checks,
    name="Đạt yêu cầu",
).to_frame()
h3_season_summary_vi = h3_season_summary.rename_axis(
    "Nhóm mùa"
).rename(
    columns={
        "movie_count": "Số phim",
        "mean_revenue": "Doanh thu trung bình",
        "median_revenue": "Doanh thu trung vị",
        "revenue_std": "Độ lệch chuẩn doanh thu",
        "minimum_revenue": "Doanh thu nhỏ nhất",
        "maximum_revenue": "Doanh thu lớn nhất",
    }
)
h3_trimmed_mean_vi = h3_trimmed_mean_by_group.to_frame().rename_axis(
    "Nhóm mùa"
).rename(
    columns={
        "trimmed_mean_revenue": "Doanh thu trung bình cắt ngọn"
    }
)
h3_evaluation_vi = h3_evaluation.rename(
    columns={
        "hypothesis_id": "Mã giả thuyết",
        "decision": "Kết luận",
        "mean_uplift_percent": "Mức tăng trung bình (%)",
        "required_uplift_percent": "Ngưỡng yêu cầu (%)",
        "median_uplift_percent": "Mức tăng trung vị (%)",
        "trimmed_mean_uplift_percent": "Mức tăng trung bình cắt ngọn (%)",
        "offpeak_movie_count": "Số phim Thấp điểm",
        "peak_movie_count": "Số phim Cao điểm",
        "reason": "Lý do",
    }
)

print("=== KIỂM TRA ĐẦU VÀO TỪ FR04 ===")
display(h3_input_checks_vi)
print("\n=== BẢNG KẾT QUẢ PHÂN TÍCH GIẢ THUYẾT 3 ===")
display(h3_season_summary_vi)
print("\n=== KIỂM TRA ĐỘ NHẠY CỦA GIẢ THUYẾT 3 ===")
display(h3_trimmed_mean_vi)
print("\n=== ĐÁNH GIÁ GIẢ THUYẾT 3 ===")
display(h3_evaluation_vi)

# FR06 - Visualize the Evidence

## 1. Primary Visualization for Hypothesis 1
To visually validate **Hypothesis 1**, we construct a dual-panel Scatter Plot with linear Trendlines comparing:
- **Panel A**: `Log10(Vote Count)` vs `Log10(Revenue)` ($r = 0.6115$).
- **Panel B**: `Vote Average` vs `Log10(Revenue)` ($r = 0.1944$).

## 2. Visualization Standards Compliance
- **Chart Choice**: Scatter plot is used to illustrate bivariate relationships and spread.
- **Scale Optimization**: Logarithmic scale ($\log_{10}$) is applied to revenue and vote count to prevent misleading compression caused by extreme right-skewness.
- **Reproducibility**: The generated chart is saved as high-resolution image `outputs/figures/hypothesis1_correlation.png`.

In [ ]:
import sys
import os
import pandas as pd

# Ensure src directory is in sys path
sys.path.append(os.path.abspath('../src'))
from visualizer import visualize_hypothesis_1

# Load cleaned dataset from FR04
df_clean = pd.read_csv('../data/processed/dataset_clean.csv')

# Generate FR06 Scatter Plot Visualization
fig_path = visualize_hypothesis_1(df_clean, output_dir='../outputs/figures')
print(f'=== FR06 CHART SAVED SUCCESSFULLY ===')
print(f'File Location: {fig_path}')


## Giả thuyết 3: Trực quan hóa bằng chứng về mùa phát hành

### 1. Biểu đồ chính

Biểu đồ chính so sánh doanh thu trung bình của nhóm Thấp điểm và nhóm Cao điểm. Đường ngang nét đứt thể hiện mức bằng 140% doanh thu trung bình của nhóm Thấp điểm, tức ngưỡng chấp nhận đã quy định trước.

### 2. Biểu đồ bổ sung theo tháng

Biểu đồ 12 tháng giúp kiểm tra kết quả của nhóm Cao điểm có xuất hiện ở nhiều tháng hay chỉ bị một tháng riêng lẻ kéo lên. Màu sắc phân biệt tháng Cao điểm và Thấp điểm; số phim được ghi trên từng cột để người đọc nhận biết sự mất cân bằng về cỡ mẫu.

In [ ]:
import os
import sys
from pathlib import Path

import matplotlib.pyplot as plt

sys.path.append(os.path.abspath("../src"))

from hypotheses.hypothesis_3.visualizer import (
    plot_monthly_revenue,
    plot_season_revenue,
)

h3_figure_dir = Path("../outputs/figures")
h3_figure_dir.mkdir(parents=True, exist_ok=True)

h3_fig_primary, _ = plot_season_revenue(
    h3_season_summary,
    h3_figure_dir / "hypothesis3_release_season_revenue.png",
)
plt.show()

h3_fig_month, _ = plot_monthly_revenue(
    h3_month_summary,
    h3_figure_dir / "hypothesis3_monthly_revenue.png",
)
plt.show()

print(
    "Biểu đồ chính:",
    h3_figure_dir / "hypothesis3_release_season_revenue.png",
)
print(
    "Biểu đồ theo tháng:",
    h3_figure_dir / "hypothesis3_monthly_revenue.png",
)

# FR07 - Đánh giá các giả thuyết

## Giả thuyết 3: Kết luận và hạn chế

**Kết luận: Chấp nhận.** Trong tập 14.839 phim hợp lệ, doanh thu trung bình của nhóm Cao điểm cao hơn nhóm Thấp điểm khoảng **94,88%**, vượt ngưỡng 40% đã đặt trước. Hai nhóm đều vượt xa cỡ mẫu tối thiểu 30 phim và mức tăng trung bình cắt ngọn vẫn dương, khoảng **108,67%**. Vì vậy, kết luận không bị đảo chiều khi giảm ảnh hưởng của các giá trị cực đoan.

Tuy nhiên, kết quả theo thập niên không hoàn toàn đồng nhất: một số giai đoạn có mức tăng thấp hoặc âm. Doanh thu chưa được điều chỉnh theo lạm phát và phân tích chưa kiểm soát ngân sách, thể loại, quy mô phát hành, quảng bá hoặc thương hiệu. Vì vậy, kết quả chỉ mô tả mối liên hệ trong bộ dữ liệu, không chứng minh mùa phát hành gây ra doanh thu cao hơn.

In [ ]:
print("=== FR07: KẾT LUẬN GIẢ THUYẾT 3 ===")
display(h3_evaluation_vi)

assert h3_evaluation.loc[0, "decision"] in {
    "Chấp nhận",
    "Bác bỏ",
    "Chưa đủ bằng chứng",
}